# Projeto (Seção 10) — solução do instrutor

## Especificação

Foi aqui que paramos no projeto da seção anterior (solução do instrutor):

In [1]:
countries = {
    "australia": ("Australia", "AUS"),
    "canada": ("Canada", "CAN"),
    "china": ("China", "CHN"),
    "france": ("France", "FRA"),
    "germany": ("Germany", "DEU"),
    "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"),
    "norway": ("Norway", "NOR"),
    "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"),
    "sanmarino": ("San Marino", "SMR"),
    "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"),
    "united kingdom": ("United Kingdom", "GBR"),
    "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"),
    "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"),
    "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"),
}
valid_country_names = sorted(countries.keys())
valid_country_names

def lookup_country(name: str) -> tuple[str, str]:
    name = name.strip().casefold()
    
    try:
        return countries[name]
    except KeyError:
        raise ValueError(
            "Unknown country name. "
            f"Country name must be one of: {','.join(valid_country_names)}"
        )

country_code_lookup = {
    name: code
    for name, code in countries.values()
}

In [2]:
from datetime import date
from enum import Enum
from functools import cached_property
from typing import Annotated, TypeVar
from uuid import uuid4
from pydantic import (
    AfterValidator, 
    BaseModel, 
    ConfigDict, 
    computed_field,
    Field, 
    field_serializer, 
    field_validator,
    UUID4,
    ValidationInfo,
)
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


T = TypeVar('T')
BoundedString = Annotated[str, Field(min_length=2, max_length=50)]
BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]


Country = Annotated[str, AfterValidator(lambda name: lookup_country(name)[0])]


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4) 
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: date = Field(
        validation_alias="completionDate", 
        ge=date(1980, 1, 1), 
        repr=False
    )
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD", 
        serialization_alias="baseMSRPUSD",
        repr=False,
    )
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(
        default=4, 
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
        repr=False,
    )
    registration_country: Country | None = Field(default=None, repr=False)
    
    @computed_field(repr=False)
    @cached_property
    def registration_country_code(self) -> str:
        return country_code_lookup[self.registration_country] 
    
    registration_date: date | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)

    
    @field_serializer("manufactured_date", "registration_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")
        
    @field_validator("registration_date")
    @classmethod
    def validate_registration_date(cls, value:date, values: ValidationInfo):
        data = values.data
        if "manufactured_date" in data and data["manufactured_date"] > value:
            raise ValueError("Automobile cannot be registered prior to manufacture date.")
        return value

Como o nosso serializador de datas é usado mais de uma vez, um tipo `Annotated` pode ser uma abordagem melhor — tiramos essa lógica de dentro do modelo e o deixamos um pouco mais simples.

Neste exercício, crie um tipo `Annotated` para a serialização de datas e use-o no modelo. Dê ao tipo o nome `CustomDate`.

Você pode reaproveitar dados de exemplo anteriores para testar o modelo.

## Solução

In [3]:
def serialize_date(value: date) -> str:
        return value.strftime("%Y/%m/%d")

In [4]:
from pydantic import PlainSerializer

CustomDate = Annotated[
    date, 
    PlainSerializer(serialize_date, when_used="json-unless-none")
]

E atualizamos o modelo:

In [5]:
class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4) 
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: CustomDate = Field(
        validation_alias="completionDate", 
        ge=date(1980, 1, 1), 
        repr=False
    )
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD", 
        serialization_alias="baseMSRPUSD",
        repr=False,
    )
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(
        default=4, 
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
        repr=False,
    )
    registration_country: Country | None = Field(default=None, repr=False)

    @computed_field(repr=False)
    @cached_property
    def registration_country_code(self) -> str:
        return country_code_lookup[self.registration_country] 
    
    registration_date: CustomDate | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)
        
    @field_validator("registration_date")
    @classmethod
    def validate_registration_date(cls, value:date, values: ValidationInfo):
        data = values.data
        if "manufactured_date" in data and data["manufactured_date"] > value:
            raise ValueError("Automobile cannot be registered prior to manufacture date.")
        return value

Testando a desserialização:

In [6]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "us",
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

car = Automobile.model_validate(data)
print(f"car: {car!r}")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4 Competition xDrive', type_=<AutomobileType.convertible: 'Convertible'>)


E conferimos a **serialização** (tanto para `dict` quanto para JSON):

> 📝 **Nota de revisão:** o original diz "desserialização" aqui, mas o que se testa abaixo é a serialização.

In [7]:
print(f"car.model_dump(by_alias=True): {car.model_dump(by_alias=True)!r}")

car.model_dump(by_alias=True): {'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), 'manufacturer': 'BMW', 'seriesName': 'M4 Competition xDrive', 'type': <AutomobileType.convertible: 'Convertible'>, 'isElectric': False, 'manufacturedDate': datetime.date(2023, 1, 1), 'baseMSRPUSD': 93300.0, 'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'], 'vin': '1234567890', 'numberOfDoors': 2, 'registrationCountry': 'United States of America', 'registrationDate': datetime.date(2023, 6, 1), 'licensePlate': 'AAA-BBB', 'registrationCountryCode': 'USA'}


Como esperado, as datas foram serializadas como objetos `date`.

In [8]:
print(f"car.model_dump_json(by_alias=True): {car.model_dump_json(by_alias=True)!r}")

car.model_dump_json(by_alias=True): '{"id":"c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7","manufacturer":"BMW","seriesName":"M4 Competition xDrive","type":"Convertible","isElectric":false,"manufacturedDate":"2023/01/01","baseMSRPUSD":93300.0,"topFeatures":["6 cylinders","all-wheel drive","convertible"],"vin":"1234567890","numberOfDoors":2,"registrationCountry":"United States of America","registrationDate":"2023/06/01","licensePlate":"AAA-BBB","registrationCountryCode":"USA"}'


E a serialização customizada aconteceu para as duas datas na saída JSON.

> 📝 **Nota de revisão:** a solução resolve o pedido (o `@field_serializer` saiu do modelo e virou o tipo `CustomDate`), mas:
>
> 1. **Continua herdando os bugs das seções 8 e 9:** `KeyError` no `model_dump()` quando não há país; código de país cacheado fica
>    desatualizado ao trocar o país; `TypeError` no validador quando falta `registrationDate`.
> 2. **`CustomDate` é assimétrico:** serializa como `2023/06/01`, mas não sabe ler esse formato de volta — o Pydantic só aceita
>    datas ISO (`2023-06-01`). O JSON que o próprio modelo gera não volta para o modelo. Prova:

In [9]:
from pydantic import TypeAdapter, ValidationError

custom_date = TypeAdapter(CustomDate)
as_json = custom_date.dump_json(date(2023, 6, 1))
print(f"custom_date.dump_json(date(2023, 6, 1)): {as_json}")
try:
    custom_date.validate_json(as_json)
except ValidationError as ex:
    print(f"validate_json do próprio JSON: {ex.errors()[0]['msg']}")

custom_date.dump_json(date(2023, 6, 1)): b'"2023/06/01"'
validate_json do próprio JSON: Input should be a valid date or datetime, invalid date separator, expected `-`
